### Identify frequent item sets using the Apriori algorithm for a given transaction data set.

In [1]:
import pandas as pd
from mlxtend.preprocessing import TransactionEncoder
from mlxtend.frequent_patterns import apriori, association_rules

In [2]:
filename = "data_exp6.csv"

# low_memory=False avoids the mixed-dtype warning on the BillNo column
df = pd.read_csv(filename, encoding="unicode_escape", low_memory=False)

# The CSV is saved with a UTF-8 BOM, which pandas keeps attached to the
# first column name (turning it into "ï»¿BillNo"). Strip the BOM/whitespace
# from all column names so we can refer to columns normally (e.g. "BillNo").
df.columns = df.columns.str.replace("ï»¿", "", regex=False).str.strip()

print("Successfully read the CSV file.")
print(df.columns.tolist())

Successfully read the CSV file.
['BillNo', 'Itemname', 'Quantity', 'Date', 'Price', 'CustomerID', 'Country']


In [3]:
df.dropna(axis=0, subset=["CustomerID"], inplace=True)

df["BillNo"] = df["BillNo"].astype(str)
df = df[~df["BillNo"].str.contains("C")]

df["Itemname"] = df["Itemname"].astype(str).str.strip()

print(df.head())

      BillNo                            Itemname  Quantity              Date  \
45    536370                             POSTAGE       3.0  01.12.2010 08:45   
237   536392  RUSTIC  SEVENTEEN DRAWER SIDEBOARD       1.0  01.12.2010 10:29   
377   536403                             POSTAGE       1.0  01.12.2010 11:27   
1113  536527                             POSTAGE       1.0  01.12.2010 13:04   
4348  536779                        Bank Charges       1.0  02.12.2010 15:08   

      Price  CustomerID         Country  
45     18.0     12583.0          France  
237   165.0     13705.0  United Kingdom  
377    15.0     12791.0     Netherlands  
1113   18.0     12662.0         Germany  
4348   15.0     15823.0  United Kingdom  


In [4]:
transactions = df.groupby("BillNo")["Itemname"].apply(list).tolist()

print("Total transactions:", len(transactions))

Total transactions: 1471


In [5]:
te = TransactionEncoder()
te_ary = te.fit(transactions).transform(transactions)

df_encoded = pd.DataFrame(te_ary, columns=te.columns_)

print(df_encoded.head())

   36 FOIL STAR CAKE CASES  ADVENT CALENDAR GINGHAM SACK  \
0                    False                         False   
1                    False                         False   
2                    False                         False   
3                    False                         False   
4                    False                         False   

   ASSTD DESIGN 3D PAPER STICKERS  BEADED CHANDELIER T-LIGHT HOLDER  \
0                           False                             False   
1                           False                             False   
2                           False                             False   
3                           False                             False   
4                           False                             False   

   BILI NUT AND WOOD NECKLACE  BISCUIT TIN VINTAGE CHRISTMAS  \
0                       False                          False   
1                       False                          False   
2                   

In [8]:
# NOTE: min_support was lowered from 0.01 to 0.002. At 0.01 every
# frequent itemset found was a single item, so no association rules
# (which need itemsets of size >= 2) could ever be generated below.
frequent_itemsets = apriori(
    df_encoded,
    min_support=0.002,
    use_colnames=True
)

print(frequent_itemsets)

rules = association_rules(
    frequent_itemsets,
    metric="confidence",
    min_threshold=0.5
)

print(rules)

      support                                           itemsets
0    0.002039      frozenset({BEADED CHANDELIER T-LIGHT HOLDER})
1    0.002719            frozenset({BILI NUT AND WOOD NECKLACE})
2    0.042828          frozenset({BOTANICAL GARDENS WALL CLOCK})
3    0.004079     frozenset({BROWN KUKUI COCONUT SEED NECKLACE})
4    0.007478                          frozenset({Bank Charges})
..        ...                                                ...
544  0.002039  frozenset({DOORMAT RESPECTABLE HOUSE, DOORMAT ...
545  0.002039  frozenset({DOORMAT RESPECTABLE HOUSE, DOORMAT ...
546  0.002039  frozenset({DOORMAT RESPECTABLE HOUSE, DOORMAT ...
547  0.002039  frozenset({DOORMAT RESPECTABLE HOUSE, DOORMAT ...
548  0.002039  frozenset({DOORMAT RESPECTABLE HOUSE, DOORMAT ...

[549 rows x 2 columns]
                                           antecedents  \
0        frozenset({BEADED CHANDELIER T-LIGHT HOLDER})   
1       frozenset({FLOWERS CHANDELIER T-LIGHT HOLDER})   
2         frozenset({I